# 43｜从零实现 T5：Span Corruption、相对位置偏置与 Encoder–Decoder

本 Notebook 不调用 transformers、nn.Transformer 或 nn.MultiheadAttention，而是用基础 PyTorch 手写 T5 风格的 RMSNorm、相对位置 bucket、多头自注意力、交叉注意力、EncoderBlock、DecoderBlock 与完整 forward。输入端把连续 span 替换为 sentinel，目标端按 sentinel 顺序还原被遮盖文本。

核心不是追求大模型效果，而是建立可审计合同：缩放必须是 \(1/\sqrt{d_h}\)，decoder 不能读取未来 token，padding 不得改变有效输出，teacher forcing 的右移必须与 loss 对齐，shared embedding 必须真正共享同一个 Parameter。

> 边界：全程 CPU、离线、单线程；合成语料上的受控记忆只证明代码链路可训练，不代表真实 T5 的迁移、生成或语言能力。

## 1. 张量、掩码与复杂度合同

- encoder 输入、decoder 输入均为 [B,T] 的 long；mask 为同形 bool，True 表示有效位置。
- hidden 为 [B,T,D]，拆头后为 [B,H,T,d_h]，要求 D 能被 H 整除。
- encoder self-attention 双向；decoder self-attention 同时使用 causal mask 与 padding mask；cross-attention 的 key/value 来自 encoder。
- 每层注意力时间与显存主项是 O(B·H·T²)，FFN 主项是 O(B·T·D·D_ff)。
- sentinel 是有顺序、不可与普通 token 混用的特殊符号；span 必须非空、互不重叠并按起点排序。

In [ ]:
import copy  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import warnings  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。

warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED43 = 4307  # 计算并保存当前步骤的中间状态。
random.seed(SEED43)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED43)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE43 = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

PAD43, BOS43, EOS43, UNK43 = 0, 1, 2, 3  # 计算并保存当前步骤的中间状态。
SENTINELS43 = [4, 5, 6]  # 计算并保存当前步骤的中间状态。
VOCAB43 = [  # 计算并保存当前步骤的中间状态。
    "<pad>", "<bos>", "<eos>", "<unk>",  # 执行当前语句以推进本节示例。
    "<extra_id_0>", "<extra_id_1>", "<extra_id_2>",  # 执行当前语句以推进本节示例。
    "我", "爱", "机器", "学习", "图", "视觉", "检索", "知识", "文本",  # 执行当前语句以推进本节示例。
    "生成", "理解", "模型", "数据", "搜索", "语言", "系统",  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
TOKEN_TO_ID43 = {token: index for index, token in enumerate(VOCAB43)}  # 计算并保存当前步骤的中间状态。
SPECIAL43 = {PAD43, BOS43, EOS43, UNK43, *SENTINELS43}  # 计算并保存当前步骤的中间状态。

assert DEVICE43.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert len(VOCAB43) == len(TOKEN_TO_ID43)  # 用受控断言验证关键不变量。
assert SENTINELS43 == list(range(4, 7))  # 用受控断言验证关键不变量。
assert SPECIAL43.isdisjoint(set(range(7, len(VOCAB43))))  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE43), "vocab": len(VOCAB43)})  # 执行当前语句以推进本节示例。

## 2. Span corruption：输入压缩，目标按 sentinel 展开

若原序列为 A B C D E，遮盖 [B,C] 与 [E]，encoder 看到 A S0 D S1；target 是 S0 B C S1 E EOS。与逐 token MLM 不同，一个 sentinel 代表一个连续 span，因此 encoder 序列通常变短。

训练数据必须先切分再做随机 corruption，否则同一原文的不同遮盖版本可能跨 train/test，造成近重复泄漏。这里使用固定 span 是为了让 oracle 完全可复现。

`sentinel_ids` 虽然是可注入参数，但实际使用的前 `span_count` 个值必须数量充足、彼此唯一且来自冻结 `SENTINELS43`。不能拿 EOS 等其他 special 或普通 token 充当 sentinel，否则目标无法无歧义地还原 span。


In [ ]:
def span_corrupt43(token_ids, spans, sentinel_ids=SENTINELS43):  # 定义本节可复用的核心函数。
    token_ids = [int(x) for x in token_ids]  # 计算并保存当前步骤的中间状态。
    spans = [(int(start), int(end)) for start, end in spans]  # 计算并保存当前步骤的中间状态。
    sentinel_ids = [int(x) for x in sentinel_ids]  # 计算并保存当前步骤的中间状态。
    if len(spans) > len(sentinel_ids):  # 按当前条件选择后续控制路径。
        raise ValueError("span 数超过 sentinel 数")  # 遇到非法合同立即显式失败。
    used_sentinels = sentinel_ids[:len(spans)]  # 计算并保存当前步骤的中间状态。
    if len(set(used_sentinels)) != len(used_sentinels):  # 按当前条件选择后续控制路径。
        raise ValueError("每个实际 span 必须使用唯一 sentinel")  # 遇到非法合同立即显式失败。
    if any(sentinel not in SENTINELS43 for sentinel in used_sentinels):  # 按当前条件选择后续控制路径。
        raise ValueError("sentinel 必须来自冻结 sentinel 集合，不能使用其他 special/普通 token")  # 遇到非法合同立即显式失败。
    previous_end = 0  # 计算并保存当前步骤的中间状态。
    for index, (start, end) in enumerate(spans):  # 遍历输入元素以累积或检查结果。
        if not (0 <= start < end <= len(token_ids)):  # 按当前条件选择后续控制路径。
            raise ValueError("span 必须是合法的非空半开区间")  # 遇到非法合同立即显式失败。
        if index and start < previous_end:  # 按当前条件选择后续控制路径。
            raise ValueError("span 必须有序且不能重叠")  # 遇到非法合同立即显式失败。
        previous_end = end  # 计算并保存当前步骤的中间状态。
    if any(token in SPECIAL43 for token in token_ids):  # 按当前条件选择后续控制路径。
        raise ValueError("原始正文不能预先包含特殊 token")  # 遇到非法合同立即显式失败。

    source, target, cursor = [], [], 0  # 计算并保存当前步骤的中间状态。
    for sentinel, (start, end) in zip(sentinel_ids, spans):  # 遍历输入元素以累积或检查结果。
        source.extend(token_ids[cursor:start])  # 执行当前语句以推进本节示例。
        source.append(sentinel)  # 执行当前语句以推进本节示例。
        target.append(sentinel)  # 执行当前语句以推进本节示例。
        target.extend(token_ids[start:end])  # 执行当前语句以推进本节示例。
        cursor = end  # 计算并保存当前步骤的中间状态。
    source.extend(token_ids[cursor:])  # 执行当前语句以推进本节示例。
    target.append(EOS43)  # 执行当前语句以推进本节示例。
    return source, target  # 返回当前分支计算出的结果。


probe_raw43 = [7, 8, 9, 10, 11]  # 计算并保存当前步骤的中间状态。
probe_source43, probe_target43 = span_corrupt43(probe_raw43, [(1, 3), (4, 5)])  # 计算并保存当前步骤的中间状态。
assert probe_source43 == [7, 4, 10, 5]  # 用受控断言验证关键不变量。
assert probe_target43 == [4, 8, 9, 5, 11, EOS43]  # 用受控断言验证关键不变量。
assert len(probe_source43) < len(probe_raw43)  # 用受控断言验证关键不变量。
assert probe_target43.count(4) == 1 and probe_target43.count(5) == 1  # 用受控断言验证关键不变量。

for bad_spans in [[(2, 2)], [(2, 4), (3, 5)], [(-1, 2)]]:  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        span_corrupt43(probe_raw43, bad_spans)  # 执行当前语句以推进本节示例。
        raise AssertionError("非法 span 未被拒绝")  # 遇到非法合同立即显式失败。
    except ValueError:  # 捕获预期异常并验证失败分支。
        pass  # 调整当前循环或占位控制流。

for bad_sentinels43 in [[4], [4, 4], [4, EOS43], [4, 7]]:  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        span_corrupt43(probe_raw43, [(1, 2), (3, 4)], sentinel_ids=bad_sentinels43)  # 计算并保存当前步骤的中间状态。
        raise AssertionError("不足、重复或非冻结 sentinel 未被拒绝")  # 遇到非法合同立即显式失败。
    except ValueError:  # 捕获预期异常并验证失败分支。
        pass  # 调整当前循环或占位控制流。

RAW_RECORDS43 = [  # 计算并保存当前步骤的中间状态。
    {"id": "r0", "tokens": [7, 8, 9, 10, 21, 18], "spans": [(2, 4)]},  # 执行当前语句以推进本节示例。
    {"id": "r1", "tokens": [11, 12, 17, 18, 19, 22], "spans": [(1, 3), (4, 5)]},  # 执行当前语句以推进本节示例。
    {"id": "r2", "tokens": [15, 13, 20, 22, 19, 18], "spans": [(0, 2)]},  # 执行当前语句以推进本节示例。
    {"id": "r3", "tokens": [14, 11, 18, 17, 16, 21], "spans": [(2, 4), (5, 6)]},  # 执行当前语句以推进本节示例。
    {"id": "r4", "tokens": [20, 13, 19, 22, 9, 10], "spans": [(1, 2), (4, 6)]},  # 执行当前语句以推进本节示例。
    {"id": "r5", "tokens": [21, 18, 16, 15, 17, 22], "spans": [(2, 5)]},  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
EXAMPLES43 = []  # 计算并保存当前步骤的中间状态。
for record in RAW_RECORDS43:  # 遍历输入元素以累积或检查结果。
    source, target = span_corrupt43(record["tokens"], record["spans"])  # 计算并保存当前步骤的中间状态。
    EXAMPLES43.append({**record, "source": source, "target": target})  # 执行当前语句以推进本节示例。

assert len({record["id"] for record in EXAMPLES43}) == len(EXAMPLES43)  # 用受控断言验证关键不变量。
assert all(example["target"][-1] == EOS43 for example in EXAMPLES43)  # 用受控断言验证关键不变量。
assert all(set(example["source"]).isdisjoint({BOS43, EOS43}) for example in EXAMPLES43)  # 用受控断言验证关键不变量。

## 3. RMSNorm 与 T5 相对位置 bucket

RMSNorm 计算 x / sqrt(mean(x²)+eps) 后乘逐维权重，不减均值。相对位置偏置不保存绝对位置 embedding，而是把 key_position-query_position 映射进有限 bucket：近距离逐格保留，远距离按对数压缩。

双向 encoder 把 bucket 一半分给左侧、一半分给右侧；单向 decoder 只编码当前与过去，未来位置最终还会被 causal mask 设为负无穷。max_distance 只影响远距离压缩，并不扩大可见范围。

In [ ]:
def relative_position_bucket43(relative_position, bidirectional, num_buckets=8, max_distance=16):  # 定义本节可复用的核心函数。
    if num_buckets < 4 or max_distance <= num_buckets // 2:  # 按当前条件选择后续控制路径。
        raise ValueError("bucket 配置过小")  # 遇到非法合同立即显式失败。
    relative_position = torch.as_tensor(relative_position, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    n = -relative_position  # 计算并保存当前步骤的中间状态。
    if bidirectional:  # 按当前条件选择后续控制路径。
        if num_buckets % 2:  # 按当前条件选择后续控制路径。
            raise ValueError("双向 bucket 数必须为偶数")  # 遇到非法合同立即显式失败。
        buckets_per_side = num_buckets // 2  # 计算并保存当前步骤的中间状态。
        sign_offset = n.lt(0).long() * buckets_per_side  # 计算并保存当前步骤的中间状态。
        n = n.abs()  # 计算并保存当前步骤的中间状态。
    else:  # 处理前置条件不成立的分支。
        buckets_per_side = num_buckets  # 计算并保存当前步骤的中间状态。
        sign_offset = torch.zeros_like(n)  # 计算并保存当前步骤的中间状态。
        n = n.clamp_min(0)  # 计算并保存当前步骤的中间状态。
    max_exact = buckets_per_side // 2  # 计算并保存当前步骤的中间状态。
    is_small = n < max_exact  # 计算并保存当前步骤的中间状态。
    safe_n = n.float().clamp_min(float(max_exact))  # 计算并保存当前步骤的中间状态。
    logarithmic = max_exact + (  # 计算并保存当前步骤的中间状态。
        torch.log(safe_n / max_exact)  # 执行当前语句以推进本节示例。
        / math.log(max_distance / max_exact)  # 执行当前语句以推进本节示例。
        * (buckets_per_side - max_exact)  # 执行当前语句以推进本节示例。
    ).long()  # 执行当前语句以推进本节示例。
    logarithmic = logarithmic.clamp(max=buckets_per_side - 1)  # 计算并保存当前步骤的中间状态。
    return sign_offset + torch.where(is_small, n, logarithmic)  # 返回当前分支计算出的结果。


class RMSNorm43(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, eps=1e-6):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.weight = nn.Parameter(torch.ones(dim))  # 计算并保存当前步骤的中间状态。
        self.eps = eps  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.shape[-1] != self.weight.numel():  # 按当前条件选择后续控制路径。
            raise ValueError("RMSNorm 最后一维不匹配")  # 遇到非法合同立即显式失败。
        rms_inverse = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)  # 计算并保存当前步骤的中间状态。
        return (x.float() * rms_inverse).to(x.dtype) * self.weight  # 返回当前分支计算出的结果。


class RelativePositionBias43(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, heads, num_buckets=8, max_distance=16, bidirectional=True):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.heads = heads  # 计算并保存当前步骤的中间状态。
        self.num_buckets = num_buckets  # 计算并保存当前步骤的中间状态。
        self.max_distance = max_distance  # 计算并保存当前步骤的中间状态。
        self.bidirectional = bidirectional  # 计算并保存当前步骤的中间状态。
        self.embedding = nn.Embedding(num_buckets, heads)  # 计算并保存当前步骤的中间状态。

    def forward(self, query_length, key_length):  # 定义本节可复用的核心函数。
        query_position = torch.arange(query_length)[:, None]  # 计算并保存当前步骤的中间状态。
        key_position = torch.arange(key_length)[None, :]  # 计算并保存当前步骤的中间状态。
        relative = key_position - query_position  # 计算并保存当前步骤的中间状态。
        buckets = relative_position_bucket43(  # 计算并保存当前步骤的中间状态。
            relative, self.bidirectional, self.num_buckets, self.max_distance  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        return self.embedding(buckets).permute(2, 0, 1).unsqueeze(0)  # 返回当前分支计算出的结果。


rms_probe43 = torch.tensor([[[3.0, 4.0]]])  # 计算并保存当前步骤的中间状态。
rms_layer43 = RMSNorm43(2, eps=0.0)  # 计算并保存当前步骤的中间状态。
rms_expected43 = rms_probe43 / math.sqrt((9.0 + 16.0) / 2.0)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(rms_layer43(rms_probe43), rms_expected43, atol=1e-7)  # 用受控断言验证关键不变量。

positions43 = torch.tensor([0, -1, -2, -16, 1, 2, 16])  # 计算并保存当前步骤的中间状态。
assert relative_position_bucket43(positions43, True).tolist() == [0, 1, 2, 3, 5, 6, 7]  # 用受控断言验证关键不变量。
causal_buckets43 = relative_position_bucket43(torch.tensor([0, -1, -2, -4, -16, 1]), False)  # 计算并保存当前步骤的中间状态。
assert causal_buckets43.tolist() == [0, 1, 2, 4, 7, 0]  # 用受控断言验证关键不变量。
assert int(relative_position_bucket43(torch.tensor(-10_000), False)) == 7  # 用受控断言验证关键不变量。

## 4. 手写多头注意力：scale、bias 与两类 mask 的先后顺序

Q、K、V 先线性投影并拆成 H 个头，score=QKᵀ/sqrt(d_h)+relative_bias。随后屏蔽无效 key，decoder self-attention 再屏蔽上三角未来位置，最后 softmax。query padding 不能只屏蔽 score：还要在输出端归零，否则投影 bias 会让 padding 重新变成非零。

In [ ]:
class ManualAttention43(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, heads):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if dim % heads:  # 按当前条件选择后续控制路径。
            raise ValueError("dim 必须整除 heads")  # 遇到非法合同立即显式失败。
        self.dim, self.heads, self.head_dim = dim, heads, dim // heads  # 计算并保存当前步骤的中间状态。
        self.q_proj = nn.Linear(dim, dim)  # 计算并保存当前步骤的中间状态。
        self.k_proj = nn.Linear(dim, dim)  # 计算并保存当前步骤的中间状态。
        self.v_proj = nn.Linear(dim, dim)  # 计算并保存当前步骤的中间状态。
        self.out_proj = nn.Linear(dim, dim)  # 计算并保存当前步骤的中间状态。

    def _split(self, x):  # 定义本节可复用的核心函数。
        batch, length, _ = x.shape  # 计算并保存当前步骤的中间状态。
        return x.view(batch, length, self.heads, self.head_dim).transpose(1, 2)  # 返回当前分支计算出的结果。

    def forward(  # 定义本节可复用的核心函数。
        self, query, key_value, key_mask, query_mask,  # 执行当前语句以推进本节示例。
        causal=False, relative_bias=None,  # 计算并保存当前步骤的中间状态。
    ):  # 执行当前语句以推进本节示例。
        if query.ndim != 3 or key_value.ndim != 3:  # 按当前条件选择后续控制路径。
            raise ValueError("query/key_value 必须是三维")  # 遇到非法合同立即显式失败。
        batch, query_length, dim = query.shape  # 计算并保存当前步骤的中间状态。
        if key_value.shape[0] != batch or key_value.shape[2] != dim or dim != self.dim:  # 按当前条件选择后续控制路径。
            raise ValueError("attention 张量形状不匹配")  # 遇到非法合同立即显式失败。
        key_length = key_value.shape[1]  # 计算并保存当前步骤的中间状态。
        if key_mask.shape != (batch, key_length) or key_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("key_mask 合同错误")  # 遇到非法合同立即显式失败。
        if query_mask.shape != (batch, query_length) or query_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("query_mask 合同错误")  # 遇到非法合同立即显式失败。
        if not bool(key_mask.any(dim=1).all()):  # 按当前条件选择后续控制路径。
            raise ValueError("每个样本至少需要一个有效 key")  # 遇到非法合同立即显式失败。
        if causal and query_length != key_length:  # 按当前条件选择后续控制路径。
            raise ValueError("本教学实现的 causal attention 要求 Q/K 等长")  # 遇到非法合同立即显式失败。

        q = self._split(self.q_proj(query))  # 计算并保存当前步骤的中间状态。
        k = self._split(self.k_proj(key_value))  # 计算并保存当前步骤的中间状态。
        v = self._split(self.v_proj(key_value))  # 计算并保存当前步骤的中间状态。
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # 计算并保存当前步骤的中间状态。
        if relative_bias is not None:  # 按当前条件选择后续控制路径。
            if relative_bias.shape != (1, self.heads, query_length, key_length):  # 按当前条件选择后续控制路径。
                raise ValueError("relative_bias 形状错误")  # 遇到非法合同立即显式失败。
            scores = scores + relative_bias.to(scores.dtype)  # 计算并保存当前步骤的中间状态。
        visible = key_mask[:, None, None, :].expand(-1, self.heads, query_length, -1)  # 计算并保存当前步骤的中间状态。
        if causal:  # 按当前条件选择后续控制路径。
            visible = visible & torch.ones(  # 计算并保存当前步骤的中间状态。
                query_length, key_length, dtype=torch.bool, device=query.device  # 计算并保存当前步骤的中间状态。
            ).tril()[None, None]  # 执行当前语句以推进本节示例。
        weights = torch.softmax(scores.masked_fill(~visible, -torch.inf), dim=-1)  # 计算并保存当前步骤的中间状态。
        weights = weights * query_mask[:, None, :, None]  # 计算并保存当前步骤的中间状态。
        context = torch.matmul(weights, v).transpose(1, 2).contiguous().view(batch, query_length, dim)  # 计算并保存当前步骤的中间状态。
        output = self.out_proj(context) * query_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return output, weights  # 返回当前分支计算出的结果。


attention_probe43 = ManualAttention43(4, 2)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    for layer in [  # 遍历输入元素以累积或检查结果。
        attention_probe43.q_proj, attention_probe43.k_proj,  # 执行当前语句以推进本节示例。
        attention_probe43.v_proj, attention_probe43.out_proj,  # 执行当前语句以推进本节示例。
    ]:  # 执行当前语句以推进本节示例。
        layer.weight.copy_(torch.eye(4))  # 执行当前语句以推进本节示例。
        layer.bias.zero_()  # 执行当前语句以推进本节示例。
x_probe43 = torch.tensor([[[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]]])  # 计算并保存当前步骤的中间状态。
mask_probe43 = torch.ones(1, 2, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
_, weights_probe43 = attention_probe43(  # 计算并保存当前步骤的中间状态。
    x_probe43, x_probe43, mask_probe43, mask_probe43, causal=False  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
expected_head0_scores43 = torch.tensor([[1.0, 0.0], [0.0, 1.0]]) / math.sqrt(2.0)  # 计算并保存当前步骤的中间状态。
expected_head0_weights43 = torch.softmax(expected_head0_scores43, dim=-1)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(weights_probe43[0, 0], expected_head0_weights43, atol=1e-7)  # 用受控断言验证关键不变量。
assert torch.allclose(weights_probe43[0, 1], torch.full((2, 2), 0.5), atol=1e-7)  # 用受控断言验证关键不变量。

_, causal_weights43 = attention_probe43(  # 计算并保存当前步骤的中间状态。
    x_probe43, x_probe43, mask_probe43, mask_probe43, causal=True  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert causal_weights43[0, :, 0, 1].abs().max().item() == 0.0  # 用受控断言验证关键不变量。
query_padding43 = torch.tensor([[True, False]])  # 计算并保存当前步骤的中间状态。
padded_output43, padded_weights43 = attention_probe43(  # 计算并保存当前步骤的中间状态。
    x_probe43, x_probe43, mask_probe43, query_padding43  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
assert padded_output43[0, 1].abs().max().item() == 0.0  # 用受控断言验证关键不变量。
assert padded_weights43[0, :, 1].abs().max().item() == 0.0  # 用受控断言验证关键不变量。

## 5. EncoderBlock、DecoderBlock 与 tied vocabulary projection

每个子层采用 pre-norm：RMSNorm 后进入 attention/FFN，再与残差相加。decoder 依次执行 causal self-attention、encoder-decoder cross-attention、FFN。这里为了清楚省略 dropout。

T5 的 encoder embedding、decoder embedding 与输出词表矩阵共享参数。forward 中直接使用 F.linear(hidden, shared.weight)，因此不是复制数值，而是同一 Parameter 同时接收输入端与输出端梯度。

In [ ]:
class FeedForward43(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, hidden_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.up = nn.Linear(dim, hidden_dim, bias=False)  # 计算并保存当前步骤的中间状态。
        self.down = nn.Linear(hidden_dim, dim, bias=False)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.down(F.gelu(self.up(x)))  # 返回当前分支计算出的结果。


class EncoderBlock43(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, heads, hidden_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.norm1 = RMSNorm43(dim)  # 计算并保存当前步骤的中间状态。
        self.attention = ManualAttention43(dim, heads)  # 计算并保存当前步骤的中间状态。
        self.norm2 = RMSNorm43(dim)  # 计算并保存当前步骤的中间状态。
        self.ff = FeedForward43(dim, hidden_dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, x, mask, relative_bias):  # 定义本节可复用的核心函数。
        update, _ = self.attention(  # 计算并保存当前步骤的中间状态。
            self.norm1(x), self.norm1(x), mask, mask,  # 执行当前语句以推进本节示例。
            causal=False, relative_bias=relative_bias,  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
        x = (x + update) * mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        x = (x + self.ff(self.norm2(x))) * mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return x  # 返回当前分支计算出的结果。


class DecoderBlock43(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, heads, hidden_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.norm1 = RMSNorm43(dim)  # 计算并保存当前步骤的中间状态。
        self.self_attention = ManualAttention43(dim, heads)  # 计算并保存当前步骤的中间状态。
        self.norm2 = RMSNorm43(dim)  # 计算并保存当前步骤的中间状态。
        self.cross_attention = ManualAttention43(dim, heads)  # 计算并保存当前步骤的中间状态。
        self.norm3 = RMSNorm43(dim)  # 计算并保存当前步骤的中间状态。
        self.ff = FeedForward43(dim, hidden_dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, x, decoder_mask, memory, memory_mask, relative_bias):  # 定义本节可复用的核心函数。
        update, _ = self.self_attention(  # 计算并保存当前步骤的中间状态。
            self.norm1(x), self.norm1(x), decoder_mask, decoder_mask,  # 执行当前语句以推进本节示例。
            causal=True, relative_bias=relative_bias,  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
        x = (x + update) * decoder_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        update, _ = self.cross_attention(  # 计算并保存当前步骤的中间状态。
            self.norm2(x), memory, memory_mask, decoder_mask, causal=False  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
        x = (x + update) * decoder_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        x = (x + self.ff(self.norm3(x))) * decoder_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return x  # 返回当前分支计算出的结果。


class TinyT543(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size, dim=24, heads=4, hidden_dim=48, layers=1):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.config = {  # 计算并保存当前步骤的中间状态。
            "vocab_size": vocab_size, "dim": dim, "heads": heads,  # 执行当前语句以推进本节示例。
            "hidden_dim": hidden_dim, "layers": layers,  # 执行当前语句以推进本节示例。
        }  # 执行当前语句以推进本节示例。
        self.shared = nn.Embedding(vocab_size, dim, padding_idx=PAD43)  # 计算并保存当前步骤的中间状态。
        self.encoder_bias = RelativePositionBias43(heads, bidirectional=True)  # 计算并保存当前步骤的中间状态。
        self.decoder_bias = RelativePositionBias43(heads, bidirectional=False)  # 计算并保存当前步骤的中间状态。
        self.encoder = nn.ModuleList(  # 计算并保存当前步骤的中间状态。
            [EncoderBlock43(dim, heads, hidden_dim) for _ in range(layers)]  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        self.decoder = nn.ModuleList(  # 计算并保存当前步骤的中间状态。
            [DecoderBlock43(dim, heads, hidden_dim) for _ in range(layers)]  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        self.encoder_norm = RMSNorm43(dim)  # 计算并保存当前步骤的中间状态。
        self.decoder_norm = RMSNorm43(dim)  # 计算并保存当前步骤的中间状态。

    def encode(self, source_ids, source_mask):  # 定义本节可复用的核心函数。
        if source_ids.dtype != torch.long or source_ids.shape != source_mask.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("source 合同错误")  # 遇到非法合同立即显式失败。
        if source_mask.dtype != torch.bool or not bool(source_mask.any(dim=1).all()):  # 按当前条件选择后续控制路径。
            raise ValueError("每行 source 至少一个有效 token")  # 遇到非法合同立即显式失败。
        hidden = self.shared(source_ids) * source_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        bias = self.encoder_bias(source_ids.shape[1], source_ids.shape[1])  # 计算并保存当前步骤的中间状态。
        for block in self.encoder:  # 遍历输入元素以累积或检查结果。
            hidden = block(hidden, source_mask, bias)  # 计算并保存当前步骤的中间状态。
        return self.encoder_norm(hidden) * source_mask.unsqueeze(-1)  # 返回当前分支计算出的结果。

    def forward(self, source_ids, source_mask, decoder_ids, decoder_mask):  # 定义本节可复用的核心函数。
        if decoder_ids.dtype != torch.long or decoder_ids.shape != decoder_mask.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("decoder 合同错误")  # 遇到非法合同立即显式失败。
        if decoder_mask.dtype != torch.bool or not bool(decoder_mask.any(dim=1).all()):  # 按当前条件选择后续控制路径。
            raise ValueError("每行 decoder 至少一个有效 token")  # 遇到非法合同立即显式失败。
        memory = self.encode(source_ids, source_mask)  # 计算并保存当前步骤的中间状态。
        hidden = self.shared(decoder_ids) * decoder_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        bias = self.decoder_bias(decoder_ids.shape[1], decoder_ids.shape[1])  # 计算并保存当前步骤的中间状态。
        for block in self.decoder:  # 遍历输入元素以累积或检查结果。
            hidden = block(hidden, decoder_mask, memory, source_mask, bias)  # 计算并保存当前步骤的中间状态。
        hidden = self.decoder_norm(hidden) * decoder_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return F.linear(hidden, self.shared.weight)  # 返回当前分支计算出的结果。


model43 = TinyT543(len(VOCAB43)).to(DEVICE43)  # 计算并保存当前步骤的中间状态。
source_probe43 = torch.tensor([[7, 4, 10]], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
source_mask_probe43 = torch.ones_like(source_probe43, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
decoder_probe43 = torch.tensor([[BOS43, 4, 8]], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
decoder_mask_probe43 = torch.ones_like(decoder_probe43, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
logits_probe43 = model43(  # 计算并保存当前步骤的中间状态。
    source_probe43, source_mask_probe43, decoder_probe43, decoder_mask_probe43  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
assert logits_probe43.shape == (1, 3, len(VOCAB43))  # 用受控断言验证关键不变量。
assert sum(parameter is model43.shared.weight for parameter in model43.parameters()) == 1  # 用受控断言验证关键不变量。

decoder_changed43 = decoder_probe43.clone()  # 计算并保存当前步骤的中间状态。
decoder_changed43[0, 2] = 15  # 计算并保存当前步骤的中间状态。
logits_changed43 = model43(  # 计算并保存当前步骤的中间状态。
    source_probe43, source_mask_probe43, decoder_changed43, decoder_mask_probe43  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
assert torch.allclose(logits_probe43[:, :2], logits_changed43[:, :2], atol=1e-6)  # 用受控断言验证关键不变量。
assert not torch.allclose(logits_probe43[:, 2], logits_changed43[:, 2])  # 用受控断言验证关键不变量。

## 6. Teacher forcing、右移与 token 平均 loss

target 为 [S0, span..., EOS]。decoder 输入必须右移成 [BOS, S0, span...]；第 t 个 logits 预测 target[t]。padding 位置用 ignore_index 排除，分母是有效 target token 数，不是 B×T。

推理时没有真实前缀，只能从 BOS 开始反复前向并取最后一个位置；greedy decode 是确定性的教学基线，真实系统还会使用 beam search、长度惩罚、缓存与约束解码。

批量 greedy 的已完成样本后续写 PAD，不重复写 EOS；返回值同时包含 `tokens`、不含 BOS 的生成长度 `lengths` 和 `finished`。这样下游无需猜测首个 EOS 之后的区域，达到长度上限但未生成 EOS 的样本也能被显式识别。


In [ ]:
def pad_sequences43(sequences, pad_value=PAD43):  # 定义本节可复用的核心函数。
    max_length = max(len(sequence) for sequence in sequences)  # 计算并保存当前步骤的中间状态。
    result = torch.full((len(sequences), max_length), pad_value, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    mask = torch.zeros_like(result, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    for row, sequence in enumerate(sequences):  # 遍历输入元素以累积或检查结果。
        result[row, :len(sequence)] = torch.tensor(sequence)  # 计算并保存当前步骤的中间状态。
        mask[row, :len(sequence)] = True  # 计算并保存当前步骤的中间状态。
    return result, mask  # 返回当前分支计算出的结果。


def shift_right43(target_ids, target_mask):  # 定义本节可复用的核心函数。
    if target_ids.dtype != torch.long or target_ids.ndim != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("target_ids 必须是二维 long")  # 遇到非法合同立即显式失败。
    if target_ids.shape != target_mask.shape or target_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
        raise ValueError("target/mask 合同错误")  # 遇到非法合同立即显式失败。
    if target_ids.shape[1] == 0 or not bool(target_mask[:, 0].all()):  # 按当前条件选择后续控制路径。
        raise ValueError("每行 target 必须有非空有效前缀")  # 遇到非法合同立即显式失败。
    seen_padding = (~target_mask).cumsum(dim=1) > 0  # 计算并保存当前步骤的中间状态。
    if bool((target_mask & seen_padding).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("target_mask 必须是连续 True 前缀，只允许右 padding")  # 遇到非法合同立即显式失败。
    shifted = torch.full_like(target_ids, PAD43)  # 计算并保存当前步骤的中间状态。
    shifted[:, 0] = BOS43  # 计算并保存当前步骤的中间状态。
    shifted[:, 1:] = target_ids[:, :-1]  # 计算并保存当前步骤的中间状态。
    shifted_mask = target_mask.clone()  # 计算并保存当前步骤的中间状态。
    shifted = shifted.masked_fill(~shifted_mask, PAD43)  # 计算并保存当前步骤的中间状态。
    return shifted, shifted_mask  # 返回当前分支计算出的结果。


def sequence_loss43(logits, target_ids, target_mask):  # 定义本节可复用的核心函数。
    if logits.shape[:2] != target_ids.shape or target_ids.shape != target_mask.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("loss 形状合同错误")  # 遇到非法合同立即显式失败。
    if target_mask.dtype != torch.bool or not bool(target_mask.any()):  # 按当前条件选择后续控制路径。
        raise ValueError("至少需要一个监督 token")  # 遇到非法合同立即显式失败。
    labels = target_ids.masked_fill(~target_mask, -100)  # 计算并保存当前步骤的中间状态。
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1), ignore_index=-100)  # 返回当前分支计算出的结果。


def greedy_decode43(model, source_ids, source_mask, max_new_tokens):  # 定义本节可复用的核心函数。
    if max_new_tokens <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("max_new_tokens 必须为正")  # 遇到非法合同立即显式失败。
    if source_ids.dtype != torch.long or source_ids.ndim != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("source_ids 必须是二维 long")  # 遇到非法合同立即显式失败。
    if source_mask.shape != source_ids.shape or source_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
        raise ValueError("source_mask 合同错误")  # 遇到非法合同立即显式失败。
    generated = torch.full(  # 计算并保存当前步骤的中间状态。
        (source_ids.shape[0], 1), BOS43, dtype=torch.long, device=source_ids.device  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。
    finished = torch.zeros(source_ids.shape[0], dtype=torch.bool, device=source_ids.device)  # 计算并保存当前步骤的中间状态。
    lengths = torch.zeros(source_ids.shape[0], dtype=torch.long, device=source_ids.device)  # 计算并保存当前步骤的中间状态。
    for step in range(1, max_new_tokens + 1):  # 遍历输入元素以累积或检查结果。
        decoder_mask = generated.ne(PAD43)  # 计算并保存当前步骤的中间状态。
        proposed = model(source_ids, source_mask, generated, decoder_mask)[:, -1].argmax(-1)  # 计算并保存当前步骤的中间状态。
        active = ~finished  # 计算并保存当前步骤的中间状态。
        next_token = torch.where(active, proposed, torch.full_like(proposed, PAD43))  # 计算并保存当前步骤的中间状态。
        newly_finished = active & next_token.eq(EOS43)  # 计算并保存当前步骤的中间状态。
        lengths[newly_finished] = step  # 计算并保存当前步骤的中间状态。
        generated = torch.cat([generated, next_token[:, None]], dim=1)  # 计算并保存当前步骤的中间状态。
        finished |= newly_finished  # 计算并保存当前步骤的中间状态。
        if bool(finished.all()):  # 按当前条件选择后续控制路径。
            break  # 调整当前循环或占位控制流。
    lengths[~finished] = max_new_tokens  # 计算并保存当前步骤的中间状态。
    return {"tokens": generated, "lengths": lengths, "finished": finished}  # 返回当前分支计算出的结果。


target_oracle43 = torch.tensor([[4, 8, EOS43, PAD43]])  # 计算并保存当前步骤的中间状态。
target_mask_oracle43 = torch.tensor([[True, True, True, False]])  # 计算并保存当前步骤的中间状态。
shifted_oracle43, shifted_mask_oracle43 = shift_right43(target_oracle43, target_mask_oracle43)  # 计算并保存当前步骤的中间状态。
assert shifted_oracle43.tolist() == [[BOS43, 4, 8, PAD43]]  # 用受控断言验证关键不变量。
assert shifted_mask_oracle43.equal(target_mask_oracle43)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    shift_right43(  # 执行当前语句以推进本节示例。
        torch.tensor([[4, 8, EOS43, PAD43]]),  # 执行当前语句以推进本节示例。
        torch.tensor([[True, False, True, False]]),  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    raise AssertionError("带洞 target mask 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

class GreedyTerminationStub43:  # 定义承载本节状态与行为的数据结构。
    def __call__(self, source_ids, source_mask, decoder_ids, decoder_mask):  # 定义本节可复用的核心函数。
        batch, length = decoder_ids.shape  # 计算并保存当前步骤的中间状态。
        logits = torch.full((batch, length, len(VOCAB43)), -1_000.0)  # 计算并保存当前步骤的中间状态。
        if length == 1:  # 按当前条件选择后续控制路径。
            logits[0, -1, EOS43] = 1_000.0  # 计算并保存当前步骤的中间状态。
            logits[1, -1, 7] = 1_000.0  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            logits[:, -1, EOS43] = 1_000.0  # 计算并保存当前步骤的中间状态。
        return logits  # 返回当前分支计算出的结果。

termination_oracle43 = greedy_decode43(  # 计算并保存当前步骤的中间状态。
    GreedyTerminationStub43(), torch.tensor([[7], [8]]),  # 执行当前语句以推进本节示例。
    torch.ones(2, 1, dtype=torch.bool), max_new_tokens=5,  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert termination_oracle43["tokens"].tolist() == [  # 用受控断言验证关键不变量。
    [BOS43, EOS43, PAD43], [BOS43, 7, EOS43],  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
assert termination_oracle43["lengths"].tolist() == [1, 2]  # 用受控断言验证关键不变量。
assert termination_oracle43["finished"].tolist() == [True, True]  # 用受控断言验证关键不变量。

small_logits43 = torch.tensor([[[2.0, 0.0], [0.0, 1.0], [9.0, -9.0]]])  # 计算并保存当前步骤的中间状态。
small_target43 = torch.tensor([[0, 1, 0]])  # 计算并保存当前步骤的中间状态。
small_mask43 = torch.tensor([[True, True, False]])  # 计算并保存当前步骤的中间状态。
actual_loss43 = sequence_loss43(small_logits43, small_target43, small_mask43)  # 计算并保存当前步骤的中间状态。
manual_loss43 = (  # 计算并保存当前步骤的中间状态。
    -F.log_softmax(small_logits43[0, 0], -1)[0]  # 执行当前语句以推进本节示例。
    -F.log_softmax(small_logits43[0, 1], -1)[1]  # 执行当前语句以推进本节示例。
) / 2  # 执行当前语句以推进本节示例。
assert torch.allclose(actual_loss43, manual_loss43, atol=1e-7)  # 用受控断言验证关键不变量。

## 7. 受控训练：只验证 teacher-forcing 链路

六条固定样本全部放入一个小 batch，模型可以记忆。记录初始与最终 token loss，并在留出的记录上只检查 greedy 接口和 token 合法性。这里明确不把训练集 loss 下降称为泛化；真实评估需要独立原文、随机 corruption、序列级 exact match 与去重审计。

In [ ]:
TRAIN_IDS43, VALID_IDS43 = ["r0", "r1", "r2", "r3"], ["r4", "r5"]  # 计算并保存当前步骤的中间状态。
train_examples43 = [x for x in EXAMPLES43 if x["id"] in TRAIN_IDS43]  # 计算并保存当前步骤的中间状态。
valid_examples43 = [x for x in EXAMPLES43 if x["id"] in VALID_IDS43]  # 计算并保存当前步骤的中间状态。
train_source43, train_source_mask43 = pad_sequences43([x["source"] for x in train_examples43])  # 计算并保存当前步骤的中间状态。
train_target43, train_target_mask43 = pad_sequences43([x["target"] for x in train_examples43])  # 计算并保存当前步骤的中间状态。
train_decoder43, train_decoder_mask43 = shift_right43(train_target43, train_target_mask43)  # 计算并保存当前步骤的中间状态。

torch.manual_seed(SEED43)  # 执行当前语句以推进本节示例。
model43 = TinyT543(len(VOCAB43), dim=24, heads=4, hidden_dim=48, layers=1)  # 计算并保存当前步骤的中间状态。
optimizer43 = torch.optim.Adam(model43.parameters(), lr=0.025)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    initial_loss43 = sequence_loss43(  # 计算并保存当前步骤的中间状态。
        model43(train_source43, train_source_mask43, train_decoder43, train_decoder_mask43),  # 执行当前语句以推进本节示例。
        train_target43, train_target_mask43,  # 执行当前语句以推进本节示例。
    ).item()  # 执行当前语句以推进本节示例。

history43 = []  # 计算并保存当前步骤的中间状态。
model43.train()  # 执行当前语句以推进本节示例。
for step in range(30):  # 遍历输入元素以累积或检查结果。
    optimizer43.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    logits43 = model43(  # 计算并保存当前步骤的中间状态。
        train_source43, train_source_mask43, train_decoder43, train_decoder_mask43  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    loss43 = sequence_loss43(logits43, train_target43, train_target_mask43)  # 计算并保存当前步骤的中间状态。
    loss43.backward()  # 执行当前语句以推进本节示例。
    torch.nn.utils.clip_grad_norm_(model43.parameters(), 1.0)  # 执行当前语句以推进本节示例。
    optimizer43.step()  # 执行当前语句以推进本节示例。
    if step in {0, 9, 19, 29}:  # 按当前条件选择后续控制路径。
        history43.append((step, float(loss43.detach())))  # 执行当前语句以推进本节示例。

model43.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    final_loss43 = sequence_loss43(  # 计算并保存当前步骤的中间状态。
        model43(train_source43, train_source_mask43, train_decoder43, train_decoder_mask43),  # 执行当前语句以推进本节示例。
        train_target43, train_target_mask43,  # 执行当前语句以推进本节示例。
    ).item()  # 执行当前语句以推进本节示例。
    valid_source43, valid_source_mask43 = pad_sequences43([x["source"] for x in valid_examples43])  # 计算并保存当前步骤的中间状态。
    decoded43 = greedy_decode43(model43, valid_source43, valid_source_mask43, max_new_tokens=10)  # 计算并保存当前步骤的中间状态。
    generated43 = decoded43["tokens"]  # 计算并保存当前步骤的中间状态。

assert math.isfinite(initial_loss43) and math.isfinite(final_loss43)  # 用受控断言验证关键不变量。
assert final_loss43 < initial_loss43 * 0.35  # 用受控断言验证关键不变量。
assert generated43.shape[0] == len(VALID_IDS43)  # 用受控断言验证关键不变量。
assert generated43[:, 0].eq(BOS43).all()  # 用受控断言验证关键不变量。
assert int(generated43.min()) >= 0 and int(generated43.max()) < len(VOCAB43)  # 用受控断言验证关键不变量。
assert decoded43["lengths"].shape == (len(VALID_IDS43),)  # 用受控断言验证关键不变量。
assert bool(((decoded43["lengths"] >= 1) & (decoded43["lengths"] <= 10)).all())  # 用受控断言验证关键不变量。
print({"initial_loss": round(initial_loss43, 4), "final_loss": round(final_loss43, 4), "trace": history43})  # 执行当前语句以推进本节示例。

## 8. 不变性与失败模式

训练 loss 下降不能替代结构测试。下面分别检查：追加 masked padding 不改变原有效位置；decoder 后缀变化不影响前缀；所有参数梯度有限；非法全空 source fail closed。

常见故障包括 causal mask 方向反了、relative_position 正负号颠倒、把 query padding 只屏蔽在 key 侧、target 未右移、把 embedding 复制成独立 head，以及 span 跨切分泄漏。

In [ ]:
model43.eval()  # 执行当前语句以推进本节示例。
one_source43 = train_source43[:1, :4]  # 计算并保存当前步骤的中间状态。
one_mask43 = train_source_mask43[:1, :4]  # 计算并保存当前步骤的中间状态。
one_decoder43 = train_decoder43[:1, :4]  # 计算并保存当前步骤的中间状态。
one_decoder_mask43 = train_decoder_mask43[:1, :4]  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    base_logits43 = model43(one_source43, one_mask43, one_decoder43, one_decoder_mask43)  # 计算并保存当前步骤的中间状态。
    extended_source43 = torch.cat([one_source43, torch.tensor([[19, 20]])], dim=1)  # 计算并保存当前步骤的中间状态。
    extended_mask43 = torch.cat([one_mask43, torch.zeros(1, 2, dtype=torch.bool)], dim=1)  # 计算并保存当前步骤的中间状态。
    extended_logits43 = model43(  # 计算并保存当前步骤的中间状态。
        extended_source43, extended_mask43, one_decoder43, one_decoder_mask43  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
assert torch.allclose(base_logits43, extended_logits43, atol=2e-5)  # 用受控断言验证关键不变量。

prefix_a43 = one_decoder43.clone()  # 计算并保存当前步骤的中间状态。
prefix_b43 = one_decoder43.clone()  # 计算并保存当前步骤的中间状态。
prefix_b43[0, -1] = 12 if prefix_a43[0, -1].item() != 12 else 13  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    logits_a43 = model43(one_source43, one_mask43, prefix_a43, one_decoder_mask43)  # 计算并保存当前步骤的中间状态。
    logits_b43 = model43(one_source43, one_mask43, prefix_b43, one_decoder_mask43)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(logits_a43[:, :-1], logits_b43[:, :-1], atol=2e-5)  # 用受控断言验证关键不变量。

model43.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
gradient_loss43 = sequence_loss43(  # 计算并保存当前步骤的中间状态。
    model43(train_source43, train_source_mask43, train_decoder43, train_decoder_mask43),  # 执行当前语句以推进本节示例。
    train_target43, train_target_mask43,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
gradient_loss43.backward()  # 执行当前语句以推进本节示例。
finite_gradients43 = [  # 计算并保存当前步骤的中间状态。
    torch.isfinite(parameter.grad).all().item()  # 执行当前语句以推进本节示例。
    for parameter in model43.parameters() if parameter.grad is not None  # 遍历输入元素以累积或检查结果。
]  # 执行当前语句以推进本节示例。
assert finite_gradients43 and all(finite_gradients43)  # 用受控断言验证关键不变量。
assert model43.shared.weight.grad is not None  # 用受控断言验证关键不变量。
assert model43.shared.weight.grad.abs().sum().item() > 0  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    model43.encode(torch.tensor([[PAD43]]), torch.tensor([[False]]))  # 执行当前语句以推进本节示例。
    raise AssertionError("全空 source 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

## 9. 发布制品：包内哈希不是信任锚

调用方若能同时替换 state、manifest 和包内 hash，自签校验没有安全意义。因此 loader 先用包外、只读 publisher registry 中的已发布指纹验证整个 package，再检查内部一致性。canonical state digest 逐 key 绑定 dtype、shape 与原始 bytes；manifest 完整绑定词表、原始数据、span、split、预处理和训练 recipe。

这里的 MappingProxyType 只模拟部署侧只读配置。真实系统应由签名清单、制品仓库不可变 digest、KMS 公钥或透明日志提供信任锚。

In [ ]:
def canonical_json43(value):  # 定义本节可复用的核心函数。
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")  # 返回当前分支计算出的结果。


def canonical_state_digest43(state):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        tensor = state[key].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        header = {"key": key, "dtype": str(tensor.dtype), "shape": list(tensor.shape)}  # 计算并保存当前步骤的中间状态。
        digest.update(canonical_json43(header))  # 执行当前语句以推进本节示例。
        digest.update(tensor.numpy().tobytes(order="C"))  # 计算并保存当前步骤的中间状态。
    return digest.hexdigest()  # 返回当前分支计算出的结果。


def package_fingerprint43(package):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    digest.update(canonical_json43(package["manifest"]))  # 执行当前语句以推进本节示例。
    digest.update(canonical_state_digest43(package["state"]).encode("ascii"))  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。


manifest43 = {  # 计算并保存当前步骤的中间状态。
    "subject": "t5-span-corruption-demo@1",  # 执行当前语句以推进本节示例。
    "architecture": copy.deepcopy(model43.config),  # 执行当前语句以推进本节示例。
    "vocab": list(VOCAB43),  # 执行当前语句以推进本节示例。
    "special_ids": {  # 执行当前语句以推进本节示例。
        "pad": PAD43, "bos": BOS43, "eos": EOS43, "unk": UNK43,  # 执行当前语句以推进本节示例。
        "sentinels": list(SENTINELS43),  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
    "dataset": copy.deepcopy(RAW_RECORDS43),  # 执行当前语句以推进本节示例。
    "split": {"train": list(TRAIN_IDS43), "validation": list(VALID_IDS43)},  # 执行当前语句以推进本节示例。
    "preprocess": {  # 执行当前语句以推进本节示例。
        "tokenizer": "frozen-token-id-v1",  # 执行当前语句以推进本节示例。
        "span_format": "sorted-half-open",  # 执行当前语句以推进本节示例。
        "decoder_start": BOS43,  # 执行当前语句以推进本节示例。
        "target_suffix": EOS43,  # 执行当前语句以推进本节示例。
        "padding": "right-contiguous-mask",  # 执行当前语句以推进本节示例。
        "sentinel_policy": "unique-prefix-from-frozen-sentinel-set",  # 执行当前语句以推进本节示例。
        "greedy_finished_fill": "pad",  # 执行当前语句以推进本节示例。
        "greedy_length": "generated-token-count-excluding-bos",  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
    "recipe": {  # 执行当前语句以推进本节示例。
        "seed": SEED43, "optimizer": "Adam", "learning_rate": 0.025,  # 执行当前语句以推进本节示例。
        "steps": 30, "gradient_clip": 1.0, "objective": "teacher-forcing-token-ce",  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
state43 = {key: value.detach().cpu().clone() for key, value in model43.state_dict().items()}  # 计算并保存当前步骤的中间状态。
package43 = {  # 计算并保存当前步骤的中间状态。
    "manifest": manifest43,  # 执行当前语句以推进本节示例。
    "state": state43,  # 执行当前语句以推进本节示例。
    "internal": {  # 执行当前语句以推进本节示例。
        "manifest_digest": hashlib.sha256(canonical_json43(manifest43)).hexdigest(),  # 执行当前语句以推进本节示例。
        "state_digest": canonical_state_digest43(state43),  # 执行当前语句以推进本节示例。
    },  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
subject43 = manifest43["subject"]  # 计算并保存当前步骤的中间状态。
PUBLISHER_REGISTRY43 = MappingProxyType({subject43: package_fingerprint43(package43)})  # 计算并保存当前步骤的中间状态。


def load_published_t543(package, subject):  # 定义本节可复用的核心函数。
    if subject not in PUBLISHER_REGISTRY43:  # 按当前条件选择后续控制路径。
        raise ValueError("未知发布 subject")  # 遇到非法合同立即显式失败。
    if package_fingerprint43(package) != PUBLISHER_REGISTRY43[subject]:  # 按当前条件选择后续控制路径。
        raise ValueError("publisher registry 指纹不匹配")  # 遇到非法合同立即显式失败。
    manifest = package["manifest"]  # 计算并保存当前步骤的中间状态。
    if manifest["subject"] != subject:  # 按当前条件选择后续控制路径。
        raise ValueError("subject 不匹配")  # 遇到非法合同立即显式失败。
    if hashlib.sha256(canonical_json43(manifest)).hexdigest() != package["internal"]["manifest_digest"]:  # 按当前条件选择后续控制路径。
        raise ValueError("manifest 内部摘要不匹配")  # 遇到非法合同立即显式失败。
    if canonical_state_digest43(package["state"]) != package["internal"]["state_digest"]:  # 按当前条件选择后续控制路径。
        raise ValueError("state 内部摘要不匹配")  # 遇到非法合同立即显式失败。
    if manifest["vocab"] != VOCAB43 or manifest["special_ids"]["sentinels"] != SENTINELS43:  # 按当前条件选择后续控制路径。
        raise ValueError("词表或 sentinel 合同不匹配")  # 遇到非法合同立即显式失败。
    all_ids = {record["id"] for record in manifest["dataset"]}  # 计算并保存当前步骤的中间状态。
    train_ids = set(manifest["split"]["train"])  # 计算并保存当前步骤的中间状态。
    validation_ids = set(manifest["split"]["validation"])  # 计算并保存当前步骤的中间状态。
    if train_ids & validation_ids or train_ids | validation_ids != all_ids:  # 按当前条件选择后续控制路径。
        raise ValueError("split 必须互斥且覆盖数据")  # 遇到非法合同立即显式失败。
    for record in manifest["dataset"]:  # 遍历输入元素以累积或检查结果。
        span_corrupt43(record["tokens"], record["spans"])  # 执行当前语句以推进本节示例。
    expected_preprocess43 = {  # 计算并保存当前步骤的中间状态。
        "tokenizer": "frozen-token-id-v1",  # 执行当前语句以推进本节示例。
        "span_format": "sorted-half-open",  # 执行当前语句以推进本节示例。
        "decoder_start": BOS43,  # 执行当前语句以推进本节示例。
        "target_suffix": EOS43,  # 执行当前语句以推进本节示例。
        "padding": "right-contiguous-mask",  # 执行当前语句以推进本节示例。
        "sentinel_policy": "unique-prefix-from-frozen-sentinel-set",  # 执行当前语句以推进本节示例。
        "greedy_finished_fill": "pad",  # 执行当前语句以推进本节示例。
        "greedy_length": "generated-token-count-excluding-bos",  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    if manifest["preprocess"] != expected_preprocess43:  # 按当前条件选择后续控制路径。
        raise ValueError("预处理合同不匹配")  # 遇到非法合同立即显式失败。
    loaded = TinyT543(**manifest["architecture"])  # 计算并保存当前步骤的中间状态。
    loaded.load_state_dict(package["state"], strict=True)  # 计算并保存当前步骤的中间状态。
    loaded.eval()  # 执行当前语句以推进本节示例。
    return loaded  # 返回当前分支计算出的结果。


loaded43 = load_published_t543(package43, subject43)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    assert torch.allclose(  # 用受控断言验证关键不变量。
        loaded43(one_source43, one_mask43, one_decoder43, one_decoder_mask43),  # 执行当前语句以推进本节示例。
        model43(one_source43, one_mask43, one_decoder43, one_decoder_mask43),  # 执行当前语句以推进本节示例。
        atol=1e-7,  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。
assert canonical_state_digest43(state43) == package43["internal"]["state_digest"]  # 用受控断言验证关键不变量。

forged43 = copy.deepcopy(package43)  # 计算并保存当前步骤的中间状态。
first_key43 = sorted(forged43["state"])[0]  # 计算并保存当前步骤的中间状态。
forged43["state"][first_key43].view(-1)[0] += 1.0  # 计算并保存当前步骤的中间状态。
forged43["internal"]["state_digest"] = canonical_state_digest43(forged43["state"])  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    load_published_t543(forged43, subject43)  # 执行当前语句以推进本节示例。
    raise AssertionError("重算包内 state hash 的伪造未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as error43:  # 捕获预期异常并验证失败分支。
    assert "registry" in str(error43)  # 用受控断言验证关键不变量。

whole_replacement43 = copy.deepcopy(package43)  # 计算并保存当前步骤的中间状态。
whole_replacement43["manifest"]["recipe"]["steps"] = 71  # 计算并保存当前步骤的中间状态。
replacement_key43 = sorted(whole_replacement43["state"])[-1]  # 计算并保存当前步骤的中间状态。
whole_replacement43["state"][replacement_key43].view(-1)[-1] -= 0.25  # 计算并保存当前步骤的中间状态。
whole_replacement43["internal"]["manifest_digest"] = hashlib.sha256(  # 计算并保存当前步骤的中间状态。
    canonical_json43(whole_replacement43["manifest"])  # 执行当前语句以推进本节示例。
).hexdigest()  # 执行当前语句以推进本节示例。
whole_replacement43["internal"]["state_digest"] = canonical_state_digest43(  # 计算并保存当前步骤的中间状态。
    whole_replacement43["state"]  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    load_published_t543(whole_replacement43, subject43)  # 执行当前语句以推进本节示例。
    raise AssertionError("整体替换并重算内部 hash 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as error43:  # 捕获预期异常并验证失败分支。
    assert "registry" in str(error43)  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    PUBLISHER_REGISTRY43[subject43] = "attacker"  # 计算并保存当前步骤的中间状态。
    raise AssertionError("只读 registry 被修改")  # 遇到非法合同立即显式失败。
except TypeError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

## 10. 从教学实现到生产系统

本实现缺少大规模 tokenizer、动态 span 采样、dropout、混合精度、分布式训练、KV cache、beam search、长文本外推与安全生成策略。生产验收还应覆盖：真实数据去重、污染扫描、长度分桶、吞吐与峰值显存、不同 seed 稳定性、分布外输入、版本回滚和签名轮换。

面试中应区分三层回答：数学上解释 relative bucket 与 causal mask；代码上说明 shape 和 forward；工程上说明 split、可观测性、发布信任锚及受控实验不等于泛化。

原始资料：

- [Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer（T5）](https://jmlr.org/papers/v21/20-074.html)
- [T5 官方代码仓库](https://github.com/google-research/text-to-text-transfer-transformer)
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)